# Interpretable Invariant-Checking Anomaly Detector for SWaT Data

This notebook implements a classical, interpretable-by-construction baseline model using hand-derived physical/logical invariant rules per stage. 
It calibrates thresholds from normal data and flags anomalies when rules are violated in the attack data.



In [ ]:
# 1. Top parameter cell
STAGE_NUM = 1  # 1 to 6



In [ ]:
import pandas as pd
import numpy as np
import json
import os
import time
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

# Ensure models directory exists
os.makedirs('models', exist_ok=True)



In [ ]:
# 2. Preprocessing
def load_and_preprocess(stage):
    df_normal = pd.read_csv(f'dataset/normal_stage{stage}.csv')
    df_attack = pd.read_csv(f'dataset/attack_stage{stage}.csv')
    
    for df in [df_normal, df_attack]:
        df.columns = df.columns.str.strip()
        df['Timestamp'] = pd.to_datetime(df['Timestamp'].str.strip(), format='%d/%m/%Y %I:%M:%S %p')
        df.sort_values('Timestamp', inplace=True)
        # Forward fill and coerce numeric
        for col in df.columns:
            if col not in ['Timestamp', 'attack']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        df.ffill(inplace=True)
        df.bfill(inplace=True)
        
    return df_normal, df_attack

df_normal, df_attack = load_and_preprocess(STAGE_NUM)
print(f"Stage {STAGE_NUM} columns:", df_normal.columns.tolist())



## Physical & Logical Reasoning for Stage Invariants

This model uses domain knowledge combined with threshold calibration from normal data to formulate invariants.

- **Stage 1 (Raw Water):** 
  - *Invariant 1.1:* If the valve (MV101) is closed (1), the tank level (LIT101) should not be increasing significantly.
  - *Invariant 1.2:* Tank level LIT101 should stay strictly within the observed normal min/max bounds (with a slight margin).

- **Stage 2 (Chemical Dosing):**
  - *Invariant 2.1:* AIT202 should always be bounded by the absolute min/max of normal data.

- **Stage 3 (UF Filtration):**
  - *Invariant 3.1:* If LIT301 is extremely low (below normal 1st percentile), the inlet valve (MV301) should be open (2).
  - *Invariant 3.2:* DPIT301 (Differential Pressure) should not exceed the max normal value unless a backwash (approximated by MV304 state) is happening.

- **Stage 4 (RO Filtration):**
  - *Invariant 4.1:* LIT401 absolute bounds must not be violated.
  - *Invariant 4.2:* Flow FIT401 should match the expected range if the pump P401 is on (2).

- **Stage 5 (RO Permeate):**
  - *Invariant 5.1:* AIT504 should remain within its calibrated normal band.
  - *Invariant 5.2:* If P501 is on (2), flow FIT501 should be greater than the normal minimum operating flow.

- **Stage 6 (UF Backwash):**
  - *Invariant 6.1:* FIT601 should not deviate outside its calibrated normal band while P601/P602 are active.
  - *Invariant 6.2:* If both pumps P601 and P602 are off (1), FIT601 should be near zero (below normal 99th percentile of off-state flow).



In [ ]:
# 3. & 8. Invariant Calibration and Configuration

def calibrate_invariants(df, stage):
    thresholds = {}
    
    if stage == 1:
        if 'LIT101' in df.columns:
            thresholds['LIT101_min'] = float(df['LIT101'].quantile(0.001))
            thresholds['LIT101_max'] = float(df['LIT101'].quantile(0.999))
            
            # Rate of change threshold
            diffs = df['LIT101'].diff().dropna()
            thresholds['LIT101_max_increase'] = float(diffs[diffs > 0].quantile(0.99)) if len(diffs[diffs > 0]) > 0 else 1.0
            
    elif stage == 2:
        if 'AIT202' in df.columns:
            thresholds['AIT202_min'] = float(df['AIT202'].quantile(0.001))
            thresholds['AIT202_max'] = float(df['AIT202'].quantile(0.999))
            
    elif stage == 3:
        if 'LIT301' in df.columns:
            thresholds['LIT301_low'] = float(df['LIT301'].quantile(0.01))
        if 'DPIT301' in df.columns:
            thresholds['DPIT301_max'] = float(df['DPIT301'].quantile(0.999))
            
    elif stage == 4:
        if 'LIT401' in df.columns:
            thresholds['LIT401_max'] = float(df['LIT401'].quantile(0.999))
            thresholds['LIT401_min'] = float(df['LIT401'].quantile(0.001))
        if 'FIT401' in df.columns and 'P401' in df.columns:
            on_flow = df.loc[df['P401'] == 2, 'FIT401']
            thresholds['FIT401_min_on'] = float(on_flow.quantile(0.01)) if not on_flow.empty else 0.0
            
    elif stage == 5:
        if 'AIT504' in df.columns:
            thresholds['AIT504_min'] = float(df['AIT504'].quantile(0.001))
            thresholds['AIT504_max'] = float(df['AIT504'].quantile(0.999))
        if 'FIT501' in df.columns and 'P501' in df.columns:
            on_flow = df.loc[df['P501'] == 2, 'FIT501']
            thresholds['FIT501_min_on'] = float(on_flow.quantile(0.01)) if not on_flow.empty else 0.0
            
    elif stage == 6:
        if 'FIT601' in df.columns:
            if 'P601' in df.columns:
                off_flow = df.loc[(df['P601'] == 1) & (df.get('P602', 1) == 1), 'FIT601']
                thresholds['FIT601_max_off'] = float(off_flow.quantile(0.99)) if not off_flow.empty else 0.0
                on_flow = df.loc[(df['P601'] == 2) | (df.get('P602', 1) == 2), 'FIT601']
                thresholds['FIT601_min_on'] = float(on_flow.quantile(0.01)) if not on_flow.empty else 0.0
                thresholds['FIT601_max_on'] = float(on_flow.quantile(0.99)) if not on_flow.empty else 0.0
                
    return thresholds

calibrated_thresholds = calibrate_invariants(df_normal, STAGE_NUM)
print(f"Calibrated Thresholds for Stage {STAGE_NUM}:")
for k, v in calibrated_thresholds.items():
    print(f"  {k}: {v}")

# Save thresholds to JSON
with open(f'models/invariants_stage{STAGE_NUM}.json', 'w') as f:
    json.dump(calibrated_thresholds, f, indent=4)



In [ ]:
# 4. Invariant Function Definitions

def evaluate_invariants_stage1(row, prev_row, thresh):
    violations = []
    if prev_row is not None:
        # Invariant 1.1: If MV101 closed, LIT101 shouldn't increase significantly
        if row.get('MV101') == 1 and (row.get('LIT101', 0) - prev_row.get('LIT101', 0)) > thresh.get('LIT101_max_increase', 10):
            violations.append("LIT101 increasing while MV101 is closed")
        
    # Invariant 1.2: Absolute bounds
    if row.get('LIT101', 0) < thresh.get('LIT101_min', 0) or row.get('LIT101', 0) > thresh.get('LIT101_max', 1000):
        violations.append("LIT101 out of calibrated normal bounds")
        
    return len(violations) > 0, violations

def evaluate_invariants_stage2(row, prev_row, thresh):
    violations = []
    # Invariant 2.1: AIT202 absolute bounds
    if row.get('AIT202', 0) < thresh.get('AIT202_min', 0) or row.get('AIT202', 0) > thresh.get('AIT202_max', 100):
        violations.append("AIT202 out of calibrated normal bounds")
    return len(violations) > 0, violations

def evaluate_invariants_stage3(row, prev_row, thresh):
    violations = []
    # Invariant 3.1: If LIT301 extremely low, MV301 should open (not closed=1)
    if row.get('LIT301', 1000) < thresh.get('LIT301_low', 0) and row.get('MV301') == 1:
        violations.append("MV301 is closed while LIT301 is extremely low")
    # Invariant 3.2: DPIT301 absolute bounds (simplified for when no backwash)
    if row.get('MV304', 1) == 1 and row.get('DPIT301', 0) > thresh.get('DPIT301_max', 100):
        violations.append("DPIT301 exceeds max normal during non-backwash")
    return len(violations) > 0, violations

def evaluate_invariants_stage4(row, prev_row, thresh):
    violations = []
    # Invariant 4.1: LIT401 bounds
    if row.get('LIT401', 0) < thresh.get('LIT401_min', 0) or row.get('LIT401', 0) > thresh.get('LIT401_max', 1000):
        violations.append("LIT401 out of calibrated normal bounds")
    # Invariant 4.2: Pump 401 on -> FIT401 should be > min_on
    if row.get('P401') == 2 and row.get('FIT401', 0) < thresh.get('FIT401_min_on', 0):
        violations.append("FIT401 too low while P401 is ON")
    return len(violations) > 0, violations

def evaluate_invariants_stage5(row, prev_row, thresh):
    violations = []
    # Invariant 5.1: AIT504 bounds
    if row.get('AIT504', 0) < thresh.get('AIT504_min', 0) or row.get('AIT504', 0) > thresh.get('AIT504_max', 100):
        violations.append("AIT504 out of calibrated normal bounds")
    # Invariant 5.2: P501 on -> FIT501 > min
    if row.get('P501') == 2 and row.get('FIT501', 0) < thresh.get('FIT501_min_on', 0):
        violations.append("FIT501 too low while P501 is ON")
    return len(violations) > 0, violations

def evaluate_invariants_stage6(row, prev_row, thresh):
    violations = []
    # Invariant 6.1: FIT601 bounds during ON
    if row.get('P601') == 2 or row.get('P602') == 2:
        if row.get('FIT601', 0) < thresh.get('FIT601_min_on', 0) or row.get('FIT601', 0) > thresh.get('FIT601_max_on', 1000):
            violations.append("FIT601 out of normal bounds while P601/P602 ON")
    # Invariant 6.2: FIT601 bounds during OFF
    if row.get('P601', 1) == 1 and row.get('P602', 1) == 1:
        if row.get('FIT601', 0) > thresh.get('FIT601_max_off', 0):
            violations.append("FIT601 exceeds off-state max while P601/P602 OFF")
    return len(violations) > 0, violations

STAGE_INVARIANTS = {
    1: evaluate_invariants_stage1,
    2: evaluate_invariants_stage2,
    3: evaluate_invariants_stage3,
    4: evaluate_invariants_stage4,
    5: evaluate_invariants_stage5,
    6: evaluate_invariants_stage6
}



In [ ]:
# 5. & 7. Apply Invariants and Measure Latency

def run_inference(df, stage, thresh):
    eval_func = STAGE_INVARIANTS[stage]
    
    predictions = []
    all_violations = []
    latencies = []
    
    prev_row = None
    
    for _, row in df.iterrows():
        row_dict = row.to_dict()
        
        t0 = time.perf_counter()
        is_anomaly, violations = eval_func(row_dict, prev_row, thresh)
        t1 = time.perf_counter()
        
        latencies.append((t1 - t0) * 1000) # milliseconds
        predictions.append(int(is_anomaly))
        all_violations.append(violations)
        
        prev_row = row_dict
        
    return predictions, all_violations, latencies

y_true = df_attack['attack'].values
y_pred, violations_list, latencies = run_inference(df_attack, STAGE_NUM, calibrated_thresholds)

print(f"Mean Latency: {np.mean(latencies):.6f} ms")
print(f"p95 Latency: {np.percentile(latencies, 95):.6f} ms")



In [ ]:
# 6. Evaluation metrics and Breakdown

def evaluate_metrics(y_true, y_pred, violations_list):
    print("--- Classification Report ---")
    print(classification_report(y_true, y_pred, zero_division=0))
    
    cm = confusion_matrix(y_true, y_pred)
    print("--- Confusion Matrix ---")
    print(cm)
    
    if len(cm) == 2:
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        print(f"FPR: {fpr:.4f}")
        print(f"FNR: {fnr:.4f}")
    
    print("\n--- Breakdown of Invariant Contributions ---")
    breakdown = {}
    for idx, (is_anomaly, vios) in enumerate(zip(y_pred, violations_list)):
        actual = y_true[idx]
        for vio in vios:
            if vio not in breakdown:
                breakdown[vio] = {'TP': 0, 'FP': 0}
            if actual == 1:
                breakdown[vio]['TP'] += 1
            else:
                breakdown[vio]['FP'] += 1
                
    breakdown_df = pd.DataFrame(breakdown).T.fillna(0).astype(int)
    print(breakdown_df)
    
evaluate_metrics(y_true, y_pred, violations_list)



In [ ]:
# 9. All Stages Summary
from IPython.display import display

all_metrics = []

for s in range(1, 7):
    print(f"\nProcessing Stage {s}...")
    try:
        df_n, df_a = load_and_preprocess(s)
        thresh = calibrate_invariants(df_n, s)
        
        # Save thresholds
        with open(f'models/invariants_stage{s}.json', 'w') as f:
            json.dump(thresh, f, indent=4)
            
        y_true_s = df_a['attack'].values
        y_pred_s, vios_s, latencies_s = run_inference(df_a, s, thresh)
        
        cm = confusion_matrix(y_true_s, y_pred_s)
        if len(cm) == 2:
            tn, fp, fn, tp = cm.ravel()
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        else:
            tn, fp, fn, tp = cm[0][0], 0, 0, 0
            fpr, fnr = 0.0, 0.0
            
        prec = precision_score(y_true_s, y_pred_s, zero_division=0)
        rec = recall_score(y_true_s, y_pred_s, zero_division=0)
        f1 = f1_score(y_true_s, y_pred_s, zero_division=0)
        
        all_metrics.append({
            'Stage': s,
            'Precision': prec,
            'Recall': rec,
            'F1': f1,
            'FPR': fpr,
            'FNR': fnr,
            'Mean Latency (ms)': np.mean(latencies_s),
            'p95 Latency (ms)': np.percentile(latencies_s, 95)
        })
    except Exception as e:
        print(f"Error processing stage {s}: {e}")

summary_df = pd.DataFrame(all_metrics)
display(summary_df)

